***LastFM with BPR, Reranking with LLAMA***

In [7]:
# BPR Recommender with LLM-based Reranker for LastFM Dataset

import numpy as np
import pandas as pd
import random
from collections import defaultdict, Counter
import math
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from llama_cpp import Llama

#################################
# BPR RECOMMENDER IMPLEMENTATION
#################################

class BPRRecommender:
    def __init__(self, factors=50, learning_rate=0.01, regularization=0.01, iterations=30, random_state=42):
        """
        Bayesian Personalized Ranking (BPR) recommender algorithm
        
        Parameters:
        - factors: number of latent factors
        - learning_rate: learning rate for SGD
        - regularization: regularization parameter
        - iterations: number of training iterations
        - random_state: seed for reproducibility
        """
        self.factors = factors
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.iterations = iterations
        self.random_state = random_state
        np.random.seed(random_state)
        
    def fit(self, user_item_matrix):
        """
        Train the BPR model on the user-item matrix
        
        Parameters:
        - user_item_matrix: scipy sparse matrix with user-item interactions
        
        Returns:
        - self
        """
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        
        # Initialize latent factors
        self.user_factors = np.random.normal(0, 0.1, (self.n_users, self.factors))
        self.item_factors = np.random.normal(0, 0.1, (self.n_items, self.factors))
        
        # Create a dictionary of items each user has interacted with
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        
        # Training loop
        for iteration in range(self.iterations):
            for _ in range(user_item_matrix.nnz):
                user, pos_item, neg_item = self._sample_triplet()
                self._update_factors(user, pos_item, neg_item)
            if (iteration + 1) % 10 == 0:
                print(f"Completed iteration {iteration + 1}/{self.iterations}")
        return self
    
    def _sample_triplet(self):
        """
        Sample a (user, positive_item, negative_item) triplet for training
        """
        user = random.choice(list(self.user_items.keys()))
        pos_item = random.choice(list(self.user_items[user]))
        neg_item = random.randint(0, self.n_items - 1)
        while neg_item in self.user_items[user]:
            neg_item = random.randint(0, self.n_items - 1)
        return user, pos_item, neg_item
    
    def _update_factors(self, user, pos_item, neg_item):
        """
        Update model parameters based on a triplet
        """
        pos_pred = np.dot(self.user_factors[user], self.item_factors[pos_item])
        neg_pred = np.dot(self.user_factors[user], self.item_factors[neg_item])
        diff = neg_pred - pos_pred
        sigmoid = 1.0 / (1.0 + np.exp(-diff))
        grad_user = sigmoid * (self.item_factors[neg_item] - self.item_factors[pos_item]) + self.regularization * self.user_factors[user]
        grad_pos_item = sigmoid * (-self.user_factors[user]) + self.regularization * self.item_factors[pos_item]
        grad_neg_item = sigmoid * self.user_factors[user] + self.regularization * self.item_factors[neg_item]
        self.user_factors[user] -= self.learning_rate * grad_user
        self.item_factors[pos_item] -= self.learning_rate * grad_pos_item
        self.item_factors[neg_item] -= self.learning_rate * grad_neg_item
    
    def recommend(self, user_id, n=10, exclude_seen=True):
        """
        Generate item recommendations for a user
        
        Parameters:
        - user_id: user index
        - n: number of recommendations
        - exclude_seen: whether to exclude items the user has already interacted with
        
        Returns:
        - list of n recommended item indices
        """
        scores = np.dot(self.user_factors[user_id], self.item_factors.T)
        if exclude_seen and user_id in self.user_items:
            seen_items = list(self.user_items[user_id])
            scores[seen_items] = -np.inf
        top_items = np.argsort(scores)[::-1][:n]
        return top_items

#################################
# LLM-BASED RERANKER IMPLEMENTATION
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Item-Popularität berechnen für Novelty / Fairness
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)
        self.fallback_count = 0  # zählt Fallbacks wegen fehlender Metadaten
        self.goal_counter = Counter()  # zählt, welches Ziel wie oft gewählt wurde

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]
    
        lines = [f"{i+1}. {item['title']}\nDescription: {item['description']}" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)
    
        prompt = (
            f"[INST] A recommender system suggests the following music artists to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Based on the artist descriptions, choose the most appropriate re-ranking goal for this user:\n"
            f"- accuracy (focus on most relevant artists)\n"
            f"- diverse_first (focus on a wide variety of music genres)\n"
            f"- fair_first (focus on less popular or underrepresented artists)\n"
            f"- balance (a mix of all of the above)\n\n"
            f"Return only the goal word. [/INST]"
        )

    
        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()
    
        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                self.goal_counter[goal] += 1  # ⬅️ Logging
                return goal
    
        self.user_goal_map[user_id] = "balance"
        self.goal_counter["balance"] += 1  # ⬅️ Logging auch für Default
        return "balance"


    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)
    
        # Prüfe, wie viele empfohlene Items Metadaten enthalten
        valid_items = [
            item for item in candidates[:10]
            if self.reverse_item_mapping[item] in self.item_metadata
        ]
    
        if len(valid_items) < 5:
            # Fallback: keine LLM-Rerankings, gib Top-N Empfehlungen zurück
            return candidates[:n].tolist()
    
        # Hole Metadaten nur für die validen Items
        top_items = [
            self.item_metadata[self.reverse_item_mapping[item]]
            for item in valid_items
        ]
    
        goal = self.get_user_goal_from_llm(user_id, top_items)
    
        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))
    
        # Accuracy-Scores vorab berechnen
        user_vector = self.model.user_factors[user_id]
        predicted_scores = np.dot(user_vector, self.model.item_factors.T)
    
        selected = []
        while len(selected) < n and len(candidates) > 0:
            best_score = -np.inf
            best_item = None
    
            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = predicted_scores[item]
    
                if selected:
                    similarities = [
                        np.dot(self.model.item_factors[item], self.model.item_factors[sel_item]) /
                        (np.linalg.norm(self.model.item_factors[item]) * np.linalg.norm(self.model.item_factors[sel_item]) + 1e-10)
                        for sel_item in selected
                    ]
                    diversity_score = 1 - np.mean(similarities)
                else:
                    diversity_score = 1
    
                novelty_score = 1 - self.norm_popularity[item]
    
                combined_score = (w1 * score_accuracy +
                                  w2 * diversity_score +
                                  w3 * novelty_score)
    
                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item
    
            if best_item is None:
                break
    
            selected.append(best_item)
            candidates = candidates[candidates != best_item]
    
        return selected


#################################
# EVALUATION METRICS
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += rel / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += rel / np.log2(i + 2)
    if idcg == 0:
        return 0
    ndcg = dcg / idcg
    return ndcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    precision = num_relevant_recommended / len(recommended_items) if recommended_items else 0
    return precision

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    recall = num_relevant_recommended / len(relevant_items) if relevant_items else 0
    return recall

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = 0
        for i, count in enumerate(sorted_counts):
            cumulative_sum += (i + 1) * count
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS
#################################

def load_lastfm(path="lastfm/lastfm.inter"):
    print("Loading LastFM dataset...")
    for enc in ['utf-8','latin-1','ISO-8859-1','cp1252']:
        try:
            df = pd.read_csv(path, sep='\t',
                             names=['user_id','artist_id','weight','tag_value'],
                             encoding=enc)
            df = df.rename(columns={'artist_id': 'item_id', 'weight': 'rating'})
            df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
            df = df.dropna(subset=['rating'])
            df['rating'] = df['rating'].astype(float)
            df['timestamp'] = 1111111111
            print(f"Loaded LastFM with {len(df)} interactions")

            # ⬇️ generate_item_metadata erwartet tag_value in raw_df
            item_metadata = generate_item_metadata(df)
            return df[['user_id', 'item_id', 'rating', 'timestamp']], item_metadata
        except Exception:
            continue


def generate_item_metadata(raw_df):
    """
    Generates metadata for LastFM items based on available tags.

    Parameters:
    - raw_df: DataFrame with columns ['user_id', 'item_id', 'rating', 'timestamp', 'tag_value']

    Returns:
    - metadata: Dictionary with structure {item_id: {'title': str, 'genres': list, 'description': str}}
    """
    metadata = {}

    # Gruppiere nach item_id und kombiniere alle zugehörigen tag_value
    tag_groups = raw_df.groupby('item_id')['tag_value'].apply(lambda tags: ",".join(filter(pd.notna, tags)))

    for item_id, tags in tag_groups.items():
        tag_list = [tag.strip() for tag in tags.split(',') if tag.strip()]
        if not tag_list:
            continue  # Ignoriere Items ohne Tags
        metadata[item_id] = {
            'title': str(item_id),
            'genres': tag_list,
            'description': f"This artist is known for: {', '.join(tag_list[:10])}."  # Max 10 Tags in Prompt
        }

    return metadata


def create_user_item_matrix(ratings_df):
    """
    Creates a user-item interaction matrix in CSR format.
    
    Parameters:
    - ratings_df: DataFrame with columns ['user_id', 'item_id', 'rating', 'timestamp']
    
    Returns:
    - user_item_matrix: csr_matrix (users x items)
    - user_mapping: dict (original user_id -> internal index)
    - item_mapping: dict (original item_id -> internal index)
    """
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    
    user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_mapping), len(item_mapping)))
    
    return user_item_matrix, user_mapping, item_mapping


#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading LastFM dataset...")
    ratings_df, item_metadata = load_lastfm()
    
    print("Splitting data for evaluation...")
    value_counts = ratings_df['user_id'].value_counts()
    if value_counts.min() >= 2:
        print("Using stratified sampling...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            stratify=ratings_df['user_id'], 
            random_state=42
        )
    else:
        print("Using random sampling (some users have only 1 rating)...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            random_state=42
        )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining BPR model...")
    model = BPRRecommender(factors=20, learning_rate=0.005, regularization=0.001, iterations=30)
    model.fit(user_item_matrix)

    print("\nLoading local LLM model...")
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
        
    print("\nInitializing reranker (LLM Reranker)...")
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )

    
    rerankers = {
        "Original BPR": None,
        "LLM Reranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original BPR"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original BPR":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original BPR"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original BPR":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of all relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in item recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")

    if isinstance(llm_reranker, LLMReranker):
        print("\n" + "="*30 + " LLM RERANKER LOG " + "="*30)
        print(f"Fallbacks wegen fehlender Metadaten: {llm_reranker.fallback_count}")
        print("Zielverteilungen laut LLM:")
        for goal, count in llm_reranker.goal_counter.items():
            print(f"- {goal}: {count}x")
    
    return all_results

#################################
# MAIN EXECUTION
#################################

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading LastFM dataset...
Loading LastFM dataset...
Loaded LastFM with 92834 interactions
Splitting data for evaluation...
Using random sampling (some users have only 1 rating)...
Creating user-item matrix...

Training BPR model...
Completed iteration 10/30
Completed iteration 20/30


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Completed iteration 30/30

Loading local LLM model...

Initializing reranker (LLM Reranker)...

Evaluating 1875 users...

Evaluating Original BPR...

Evaluating LLM Reranker...

============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original BPR        LLM Reranker        
--------------------------------------------------------------------------------
ndcg@10        0.1274               0.1216 (-4.6%)     
precision@10   0.0886               0.0860 (-3.0%)     
recall@10      0.0955               0.0923 (-3.3%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original BPR        LLM Reranker        
--------------------------------------------------------------------------------
item_coverage  0.0842               0.1076 (+27.9%)     
gini_index     0.8963               0.8519 (-4.9%)     
shannon_entropy0.4597               0.5303 (+15.3%)     
tail_percentage0.0193       

_______________________________________________________________________________________________________________________________

***LastFM with ItemKNN, Reranking with LLAMA***

_______________________________________________________________________________________________________________________________

In [8]:
# ItemKNN Recommender with LLM-based Reranker for LastFM Dataset

import numpy as np
import pandas as pd
import random
from collections import defaultdict, Counter
import math
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from llama_cpp import Llama

#################################
# ITEMKNN RECOMMENDER IMPLEMENTATION
#################################

from sklearn.metrics.pairwise import cosine_similarity

class ItemKNNRecommender:
    def __init__(self, k=50, random_state=42):
        self.k = k
        self.random_state = random_state
        np.random.seed(random_state)
        
    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        print("Computing item-item similarity matrix...")
        self.item_factors = cosine_similarity(user_item_matrix.T)
        np.fill_diagonal(self.item_factors, 0)

        self.user_items = defaultdict(set)
        for user, item in zip(*user_item_matrix.nonzero()):
            self.user_items[user].add(item)

        self.item_neighbors = {}
        for item_id in range(self.n_items):
            similarities = self.item_factors[item_id]
            neighbor_ids = np.argsort(similarities)[::-1][:self.k]
            self.item_neighbors[item_id] = {
                neighbor_id: similarities[neighbor_id] 
                for neighbor_id in neighbor_ids 
                if similarities[neighbor_id] > 0
            }
        return self

    def recommend(self, user_id, n=10, exclude_seen=True):
        if user_id not in self.user_items or not self.user_items[user_id]:
            all_items = set(range(self.n_items))
            seen_items = self.user_items.get(user_id, set())
            candidate_items = list(all_items - seen_items) if exclude_seen else list(all_items)
            if len(candidate_items) <= n:
                return candidate_items
            return random.sample(candidate_items, n)
        
        scores = np.zeros(self.n_items)
        for item_id in self.user_items[user_id]:
            if item_id in self.item_neighbors:
                for neighbor_id, similarity in self.item_neighbors[item_id].items():
                    scores[neighbor_id] += similarity
        if exclude_seen:
            for item_id in self.user_items[user_id]:
                scores[item_id] = -np.inf
        top_items = np.argsort(scores)[::-1][:n]
        return top_items


#################################
# LLM-BASED RERANKER FOR ITEMKNN
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Calculate item popularity for novelty/fairness
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

        self.fallback_count = 0
        self.goal_counter = Counter()

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]

        lines = [f"{i+1}. {item['title']}\nDescription: {item['description']}" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)

        prompt = (
            f"[INST] A recommender system suggests the following music artists to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Based on the artist descriptions, choose the most appropriate re-ranking goal for this user:\n"
            f"- accuracy (focus on most relevant artists)\n"
            f"- diverse_first (focus on a wide variety of music genres)\n"
            f"- fair_first (focus on less popular or underrepresented artists)\n"
            f"- balance (a mix of all of the above)\n\n"
            f"Return only the goal word. [/INST]"
        )

        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()

        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                self.goal_counter[goal] += 1
                return goal

        self.user_goal_map[user_id] = "balance"
        self.goal_counter["balance"] += 1
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)

        # Check if enough metadata is available
        valid_items = [
            item for item in candidates[:10]
            if self.reverse_item_mapping[item] in self.item_metadata
        ]
        if len(valid_items) < 5:
            self.fallback_count += 1
            return candidates[:n].tolist()

        # Get metadata
        top_items = [
            self.item_metadata[self.reverse_item_mapping[item]]
            for item in valid_items
        ]
        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        # Compute predicted scores based on item similarity
        predicted_scores = np.zeros(self.model.n_items)
        for item in self.model.user_items.get(user_id, []):
            if item in self.model.item_neighbors:
                for neighbor, sim in self.model.item_neighbors[item].items():
                    predicted_scores[neighbor] += sim

        selected = []
        while len(selected) < n and len(candidates) > 0:
            best_score = -np.inf
            best_item = None

            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = predicted_scores[item]

                if selected:
                    similarities = [
                        np.dot(self.model.item_factors[item], self.model.item_factors[sel_item]) /
                        (np.linalg.norm(self.model.item_factors[item]) * np.linalg.norm(self.model.item_factors[sel_item]) + 1e-10)
                        for sel_item in selected
                    ]
                    diversity_score = 1 - np.mean(similarities)
                else:
                    diversity_score = 1

                novelty_score = 1 - self.norm_popularity[item]

                combined_score = (w1 * score_accuracy +
                                  w2 * diversity_score +
                                  w3 * novelty_score)

                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item

            if best_item is None:
                break

            selected.append(best_item)
            candidates = candidates[candidates != best_item]

        return selected


#################################
# EVALUATION METRICS
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += rel / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += rel / np.log2(i + 2)
    if idcg == 0:
        return 0
    ndcg = dcg / idcg
    return ndcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    precision = num_relevant_recommended / len(recommended_items) if recommended_items else 0
    return precision

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    recall = num_relevant_recommended / len(relevant_items) if relevant_items else 0
    return recall

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = 0
        for i, count in enumerate(sorted_counts):
            cumulative_sum += (i + 1) * count
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS
#################################

def load_lastfm(path="lastfm/lastfm.inter"):
    print("Loading LastFM dataset...")
    for enc in ['utf-8','latin-1','ISO-8859-1','cp1252']:
        try:
            df = pd.read_csv(path, sep='\t',
                             names=['user_id','artist_id','weight','tag_value'],
                             encoding=enc)
            df = df.rename(columns={'artist_id': 'item_id', 'weight': 'rating'})
            df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
            df = df.dropna(subset=['rating'])
            df['rating'] = df['rating'].astype(float)
            df['timestamp'] = 1111111111
            print(f"Loaded LastFM with {len(df)} interactions")

            # ⬇️ generate_item_metadata erwartet tag_value in raw_df
            item_metadata = generate_item_metadata(df)
            return df[['user_id', 'item_id', 'rating', 'timestamp']], item_metadata
        except Exception:
            continue


def generate_item_metadata(raw_df):
    """
    Generates metadata for LastFM items based on available tags.

    Parameters:
    - raw_df: DataFrame with columns ['user_id', 'item_id', 'rating', 'timestamp', 'tag_value']

    Returns:
    - metadata: Dictionary with structure {item_id: {'title': str, 'genres': list, 'description': str}}
    """
    metadata = {}

    # Gruppiere nach item_id und kombiniere alle zugehörigen tag_value
    tag_groups = raw_df.groupby('item_id')['tag_value'].apply(lambda tags: ",".join(filter(pd.notna, tags)))

    for item_id, tags in tag_groups.items():
        tag_list = [tag.strip() for tag in tags.split(',') if tag.strip()]
        if not tag_list:
            continue  # Ignoriere Items ohne Tags
        metadata[item_id] = {
            'title': str(item_id),
            'genres': tag_list,
            'description': f"This artist is known for: {', '.join(tag_list[:10])}."  # Max 10 Tags in Prompt
        }

    return metadata


#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading LastFM dataset...")
    ratings_df, item_metadata = load_lastfm()
    
    print("Splitting data for evaluation...")
    value_counts = ratings_df['user_id'].value_counts()
    if value_counts.min() >= 2:
        print("Using stratified sampling...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            stratify=ratings_df['user_id'], 
            random_state=42
        )
    else:
        print("Using random sampling (some users have only 1 rating)...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            random_state=42
        )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining ItemKNN model...")
    model = ItemKNNRecommender(k=50)
    model.fit(user_item_matrix)


    print("\nLoading local LLM model...")
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
        
    print("\nInitializing reranker (LLM Reranker)...")
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )

    
    rerankers = {
        "Original ItemKNN": None,
        "ItemKNN + LLMReranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original ItemKNN"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original ItemKNN":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original ItemKNN"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original ItemKNN":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of all relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in item recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")

    if isinstance(llm_reranker, LLMReranker):
        print("\n" + "="*30 + " LLM RERANKER LOG " + "="*30)
        print(f"Fallbacks wegen fehlender Metadaten: {llm_reranker.fallback_count}")
        print("Zielverteilungen laut LLM:")
        for goal, count in llm_reranker.goal_counter.items():
            print(f"- {goal}: {count}x")
    
    return all_results

#################################
# MAIN EXECUTION
#################################

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)


COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading LastFM dataset...
Loading LastFM dataset...
Loaded LastFM with 92834 interactions
Splitting data for evaluation...
Using random sampling (some users have only 1 rating)...
Creating user-item matrix...

Training ItemKNN model...
Computing item-item similarity matrix...


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



Loading local LLM model...

Initializing reranker (LLM Reranker)...

Evaluating 1875 users...

Evaluating Original ItemKNN...

Evaluating ItemKNN + LLMReranker...

============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original ItemKNN    ItemKNN + LLMReranker
--------------------------------------------------------------------------------
ndcg@10        0.2171               0.2166 (-0.2%)     
precision@10   0.1473               0.1460 (-0.8%)     
recall@10      0.1621               0.1609 (-0.7%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original ItemKNN    ItemKNN + LLMReranker
--------------------------------------------------------------------------------
item_coverage  0.2415               0.2622 (+8.6%)     
gini_index     0.7569               0.7350 (-2.9%)     
shannon_entropy0.6408               0.6621 (+3.3%)     
tail_percentage0.0644               0.071

***LastFM with NeuMF, Reranking with LLAMA***

In [9]:
# NeuMF Recommender with LLM-based Reranker for LastFM Dataset

import numpy as np
import pandas as pd
import random
from collections import defaultdict, Counter
import math
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from llama_cpp import Llama

# NEUMF RECOMMENDER IMPLEMENTATION
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

class NeuMFModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.gmf_user_embeddings = nn.Embedding(num_users, latent_dim)
        self.gmf_item_embeddings = nn.Embedding(num_items, latent_dim)
        self.mlp_user_embeddings = nn.Embedding(num_users, latent_dim)
        self.mlp_item_embeddings = nn.Embedding(num_items, latent_dim)
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim * 2, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )
        self.output = nn.Linear(latent_dim * 2, 1)

    def forward(self, user_indices, item_indices):
        gmf_user = self.gmf_user_embeddings(user_indices)
        gmf_item = self.gmf_item_embeddings(item_indices)
        gmf_vector = gmf_user * gmf_item

        mlp_user = self.mlp_user_embeddings(user_indices)
        mlp_item = self.mlp_item_embeddings(item_indices)
        mlp_vector = self.mlp(torch.cat([mlp_user, mlp_item], dim=-1))

        final_vector = torch.cat([gmf_vector, mlp_vector], dim=-1)
        prediction = self.output(final_vector).squeeze()
        return prediction

class NeuMFRecommender:
    def __init__(self, latent_dim=32, epochs=10, batch_size=256, lr=0.001):
        self.latent_dim = latent_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape

        user_item_pairs = list(zip(*user_item_matrix.nonzero()))
        all_items = set(range(self.n_items))
        negatives = []
        for user in range(self.n_users):
            seen = set(user_item_matrix[user].nonzero()[1])
            not_seen = list(all_items - seen)
            negatives.extend([(user, item) for item in random.sample(not_seen, len(seen))])

        user_input = [u for u, i in user_item_pairs + negatives]
        item_input = [i for u, i in user_item_pairs + negatives]
        labels = [1] * len(user_item_pairs) + [0] * len(negatives)

        dataset = torch.utils.data.TensorDataset(
            torch.LongTensor(user_input),
            torch.LongTensor(item_input),
            torch.FloatTensor(labels)
        )
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        self.model = NeuMFModel(self.n_users, self.n_items, self.latent_dim).to(self.device)
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()

        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            for users, items, labels in loader:
                users, items, labels = users.to(self.device), items.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                outputs = self.model(users, items)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            print(f"Epoch {epoch+1}/{self.epochs}, Loss: {total_loss:.4f}")

        self.item_factors = (
            self.model.gmf_item_embeddings.weight.data.cpu().numpy() +
            self.model.mlp_item_embeddings.weight.data.cpu().numpy()
        )

        self.user_items = defaultdict(set)
        for user, item in zip(*user_item_matrix.nonzero()):
            self.user_items[user].add(item)

    def recommend(self, user_id, n=10, exclude_seen=True):
        self.model.eval()
        with torch.no_grad():
            user_tensor = torch.LongTensor([user_id] * self.n_items).to(self.device)
            item_tensor = torch.LongTensor(range(self.n_items)).to(self.device)
            scores = self.model(user_tensor, item_tensor).cpu().numpy()

        if exclude_seen:
            for item in self.user_items.get(user_id, []):
                scores[item] = -np.inf
        return np.argsort(scores)[::-1][:n]



#################################
# LLM-BASED RERANKER FOR ITEMKNN
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Calculate item popularity for novelty/fairness
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

        self.fallback_count = 0
        self.goal_counter = Counter()

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]

        lines = [f"{i+1}. {item['title']}\nDescription: {item['description']}" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)

        prompt = (
            f"[INST] A recommender system suggests the following music artists to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Based on the artist descriptions, choose the most appropriate re-ranking goal for this user:\n"
            f"- accuracy (focus on most relevant artists)\n"
            f"- diverse_first (focus on a wide variety of music genres)\n"
            f"- fair_first (focus on less popular or underrepresented artists)\n"
            f"- balance (a mix of all of the above)\n\n"
            f"Return only the goal word. [/INST]"
        )

        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()

        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                self.goal_counter[goal] += 1
                return goal

        self.user_goal_map[user_id] = "balance"
        self.goal_counter["balance"] += 1
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)

        # Check if enough metadata is available
        valid_items = [
            item for item in candidates[:10]
            if self.reverse_item_mapping[item] in self.item_metadata
        ]
        if len(valid_items) < 5:
            self.fallback_count += 1
            return candidates[:n].tolist()

        # Get metadata
        top_items = [
            self.item_metadata[self.reverse_item_mapping[item]]
            for item in valid_items
        ]
        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        # Compute predicted scores using NeuMF
        self.model.model.eval()
        with torch.no_grad():
            user_tensor = torch.LongTensor([user_id] * self.model.n_items).to(self.model.device)
            item_tensor = torch.LongTensor(list(range(self.model.n_items))).to(self.model.device)
            predicted_scores = self.model.model(user_tensor, item_tensor).cpu().numpy()


        selected = []
        while len(selected) < n and len(candidates) > 0:
            best_score = -np.inf
            best_item = None

            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = predicted_scores[item]

                if selected:
                    similarities = [
                        np.dot(self.model.item_factors[item], self.model.item_factors[sel_item]) /
                        (np.linalg.norm(self.model.item_factors[item]) * np.linalg.norm(self.model.item_factors[sel_item]) + 1e-10)
                        for sel_item in selected
                    ]
                    diversity_score = 1 - np.mean(similarities)
                else:
                    diversity_score = 1

                novelty_score = 1 - self.norm_popularity[item]

                combined_score = (w1 * score_accuracy +
                                  w2 * diversity_score +
                                  w3 * novelty_score)

                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item

            if best_item is None:
                break

            selected.append(best_item)
            candidates = candidates[candidates != best_item]

        return selected


#################################
# EVALUATION METRICS
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += rel / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += rel / np.log2(i + 2)
    if idcg == 0:
        return 0
    ndcg = dcg / idcg
    return ndcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    precision = num_relevant_recommended / len(recommended_items) if recommended_items else 0
    return precision

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    recall = num_relevant_recommended / len(relevant_items) if relevant_items else 0
    return recall

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = 0
        for i, count in enumerate(sorted_counts):
            cumulative_sum += (i + 1) * count
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS
#################################

def load_lastfm(path="lastfm/lastfm.inter"):
    print("Loading LastFM dataset...")
    for enc in ['utf-8','latin-1','ISO-8859-1','cp1252']:
        try:
            df = pd.read_csv(path, sep='\t',
                             names=['user_id','artist_id','weight','tag_value'],
                             encoding=enc)
            df = df.rename(columns={'artist_id': 'item_id', 'weight': 'rating'})
            df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
            df = df.dropna(subset=['rating'])
            df['rating'] = df['rating'].astype(float)
            df['timestamp'] = 1111111111
            print(f"Loaded LastFM with {len(df)} interactions")

            # ⬇️ generate_item_metadata erwartet tag_value in raw_df
            item_metadata = generate_item_metadata(df)
            return df[['user_id', 'item_id', 'rating', 'timestamp']], item_metadata
        except Exception:
            continue


def generate_item_metadata(raw_df):
    """
    Generates metadata for LastFM items based on available tags.

    Parameters:
    - raw_df: DataFrame with columns ['user_id', 'item_id', 'rating', 'timestamp', 'tag_value']

    Returns:
    - metadata: Dictionary with structure {item_id: {'title': str, 'genres': list, 'description': str}}
    """
    metadata = {}

    # Gruppiere nach item_id und kombiniere alle zugehörigen tag_value
    tag_groups = raw_df.groupby('item_id')['tag_value'].apply(lambda tags: ",".join(filter(pd.notna, tags)))

    for item_id, tags in tag_groups.items():
        tag_list = [tag.strip() for tag in tags.split(',') if tag.strip()]
        if not tag_list:
            continue  # Ignoriere Items ohne Tags
        metadata[item_id] = {
            'title': str(item_id),
            'genres': tag_list,
            'description': f"This artist is known for: {', '.join(tag_list[:10])}."  # Max 10 Tags in Prompt
        }

    return metadata


#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading LastFM dataset...")
    ratings_df, item_metadata = load_lastfm()
    
    print("Splitting data for evaluation...")
    value_counts = ratings_df['user_id'].value_counts()
    if value_counts.min() >= 2:
        print("Using stratified sampling...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            stratify=ratings_df['user_id'], 
            random_state=42
        )
    else:
        print("Using random sampling (some users have only 1 rating)...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            random_state=42
        )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining NeuMF model...")
    model = NeuMFRecommender(latent_dim=16, epochs=10, batch_size=256)
    model.fit(user_item_matrix)



    print("\nLoading local LLM model...")
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
        
    print("\nInitializing reranker (LLM Reranker)...")
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )

    
    rerankers = {
        "Original NeuMF": None,
        "NeuMF + LLMReranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original NeuMF"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original NeuMF":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original NeuMF"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original NeuMF":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of all relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in item recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")

    if isinstance(llm_reranker, LLMReranker):
        print("\n" + "="*30 + " LLM RERANKER LOG " + "="*30)
        print(f"Fallbacks wegen fehlender Metadaten: {llm_reranker.fallback_count}")
        print("Zielverteilungen laut LLM:")
        for goal, count in llm_reranker.goal_counter.items():
            print(f"- {goal}: {count}x")
    
    return all_results

#################################
# MAIN EXECUTION
#################################

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading LastFM dataset...
Loading LastFM dataset...
Loaded LastFM with 92834 interactions
Splitting data for evaluation...
Using random sampling (some users have only 1 rating)...
Creating user-item matrix...

Training NeuMF model...
Epoch 1/10, Loss: 398.4582
Epoch 2/10, Loss: 354.6675
Epoch 3/10, Loss: 321.3454
Epoch 4/10, Loss: 300.5331
Epoch 5/10, Loss: 284.8347
Epoch 6/10, Loss: 270.7019
Epoch 7/10, Loss: 257.5788
Epoch 8/10, Loss: 244.4761
Epoch 9/10, Loss: 231.5792


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Epoch 10/10, Loss: 218.5238

Loading local LLM model...

Initializing reranker (LLM Reranker)...

Evaluating 1875 users...

Evaluating Original NeuMF...

Evaluating NeuMF + LLMReranker...

============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original NeuMF      NeuMF + LLMReranker 
--------------------------------------------------------------------------------
ndcg@10        0.0292               0.0257 (-11.8%)     
precision@10   0.0305               0.0272 (-10.7%)     
recall@10      0.0332               0.0297 (-10.4%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original NeuMF      NeuMF + LLMReranker 
--------------------------------------------------------------------------------
item_coverage  0.0170               0.0191 (+12.2%)     
gini_index     0.8634               0.8666 (+0.4%)     
shannon_entropy0.3892               0.3950 (+1.5%)     
tail_percentage

***LastFM with Pop, Reranking with LLAMA***

In [10]:
# POP Recommender with LLM-based Reranker for LastFM Dataset

import numpy as np
import pandas as pd
import random
from collections import defaultdict, Counter
import math
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from llama_cpp import Llama

# POP RECOMMENDER IMPLEMENTATION
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

class PopRecommender:
    def __init__(self, random_state=42):
        self.random_state = random_state
        random.seed(random_state)
        np.random.seed(random_state)

    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape

        self.user_items = defaultdict(set)
        for user, item in zip(*user_item_matrix.nonzero()):
            self.user_items[user].add(item)

        self.item_popularity = np.array(user_item_matrix.sum(axis=0)).flatten()

        self.item_factors = np.zeros((self.n_items, 32))
        max_pop = np.max(self.item_popularity)
        if max_pop > 0:
            self.item_factors[:, 0] = self.item_popularity / max_pop
        for i in range(self.n_items):
            np.random.seed(self.random_state + i)
            self.item_factors[i, 1:] = np.random.normal(0, 0.1, 31) * (0.5 + 0.5 * self.item_factors[i, 0])

        print("Popularity-based recommender ready! Top 5 most popular items:", 
              np.argsort(self.item_popularity)[::-1][:5])
        return self

    def recommend(self, user_id, n=10, exclude_seen=True):
        recommended_items = np.argsort(self.item_popularity)[::-1]
        if exclude_seen and user_id in self.user_items:
            seen_items = list(self.user_items[user_id])
            recommended_items = np.array([item for item in recommended_items if item not in seen_items])
        return recommended_items[:n]



#################################
# LLM-BASED RERANKER FOR ITEMKNN
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Calculate item popularity for novelty/fairness
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

        self.fallback_count = 0
        self.goal_counter = Counter()

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]

        lines = [f"{i+1}. {item['title']}\nDescription: {item['description']}" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)

        prompt = (
            f"[INST] A recommender system suggests the following music artists to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Based on the artist descriptions, choose the most appropriate re-ranking goal for this user:\n"
            f"- accuracy (focus on most relevant artists)\n"
            f"- diverse_first (focus on a wide variety of music genres)\n"
            f"- fair_first (focus on less popular or underrepresented artists)\n"
            f"- balance (a mix of all of the above)\n\n"
            f"Return only the goal word. [/INST]"
        )

        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()

        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                self.goal_counter[goal] += 1
                return goal

        self.user_goal_map[user_id] = "balance"
        self.goal_counter["balance"] += 1
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)

        # Check if enough metadata is available
        valid_items = [
            item for item in candidates[:10]
            if self.reverse_item_mapping[item] in self.item_metadata
        ]
        if len(valid_items) < 5:
            self.fallback_count += 1
            return candidates[:n].tolist()

        # Get metadata
        top_items = [
            self.item_metadata[self.reverse_item_mapping[item]]
            for item in valid_items
        ]
        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        # Compute predicted scores depending on model type
        if hasattr(self.model, 'model') and hasattr(self.model.model, 'eval'):
            # NeuMF: use neural network prediction
            self.model.model.eval()
            with torch.no_grad():
                user_tensor = torch.LongTensor([user_id] * self.model.n_items).to(self.model.device)
                item_tensor = torch.LongTensor(list(range(self.model.n_items))).to(self.model.device)
                predicted_scores = self.model.model(user_tensor, item_tensor).cpu().numpy()
        else:
            # PopRecommender: use normalized popularity as proxy for accuracy score
            predicted_scores = self.norm_popularity.copy()


        selected = []
        while len(selected) < n and len(candidates) > 0:
            best_score = -np.inf
            best_item = None

            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = predicted_scores[item]

                if selected:
                    similarities = [
                        np.dot(self.model.item_factors[item], self.model.item_factors[sel_item]) /
                        (np.linalg.norm(self.model.item_factors[item]) * np.linalg.norm(self.model.item_factors[sel_item]) + 1e-10)
                        for sel_item in selected
                    ]
                    diversity_score = 1 - np.mean(similarities)
                else:
                    diversity_score = 1

                novelty_score = 1 - self.norm_popularity[item]

                combined_score = (w1 * score_accuracy +
                                  w2 * diversity_score +
                                  w3 * novelty_score)

                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item

            if best_item is None:
                break

            selected.append(best_item)
            candidates = candidates[candidates != best_item]

        return selected


#################################
# EVALUATION METRICS
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += rel / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += rel / np.log2(i + 2)
    if idcg == 0:
        return 0
    ndcg = dcg / idcg
    return ndcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    precision = num_relevant_recommended / len(recommended_items) if recommended_items else 0
    return precision

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    recall = num_relevant_recommended / len(relevant_items) if relevant_items else 0
    return recall

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = 0
        for i, count in enumerate(sorted_counts):
            cumulative_sum += (i + 1) * count
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS
#################################

def load_lastfm(path="lastfm/lastfm.inter"):
    print("Loading LastFM dataset...")
    for enc in ['utf-8','latin-1','ISO-8859-1','cp1252']:
        try:
            df = pd.read_csv(path, sep='\t',
                             names=['user_id','artist_id','weight','tag_value'],
                             encoding=enc)
            df = df.rename(columns={'artist_id': 'item_id', 'weight': 'rating'})
            df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
            df = df.dropna(subset=['rating'])
            df['rating'] = df['rating'].astype(float)
            df['timestamp'] = 1111111111
            print(f"Loaded LastFM with {len(df)} interactions")

            # ⬇️ generate_item_metadata erwartet tag_value in raw_df
            item_metadata = generate_item_metadata(df)
            return df[['user_id', 'item_id', 'rating', 'timestamp']], item_metadata
        except Exception:
            continue


def generate_item_metadata(raw_df):
    """
    Generates metadata for LastFM items based on available tags.

    Parameters:
    - raw_df: DataFrame with columns ['user_id', 'item_id', 'rating', 'timestamp', 'tag_value']

    Returns:
    - metadata: Dictionary with structure {item_id: {'title': str, 'genres': list, 'description': str}}
    """
    metadata = {}

    # Gruppiere nach item_id und kombiniere alle zugehörigen tag_value
    tag_groups = raw_df.groupby('item_id')['tag_value'].apply(lambda tags: ",".join(filter(pd.notna, tags)))

    for item_id, tags in tag_groups.items():
        tag_list = [tag.strip() for tag in tags.split(',') if tag.strip()]
        if not tag_list:
            continue  # Ignoriere Items ohne Tags
        metadata[item_id] = {
            'title': str(item_id),
            'genres': tag_list,
            'description': f"This artist is known for: {', '.join(tag_list[:10])}."  # Max 10 Tags in Prompt
        }

    return metadata


#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading LastFM dataset...")
    ratings_df, item_metadata = load_lastfm()
    
    print("Splitting data for evaluation...")
    value_counts = ratings_df['user_id'].value_counts()
    if value_counts.min() >= 2:
        print("Using stratified sampling...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            stratify=ratings_df['user_id'], 
            random_state=42
        )
    else:
        print("Using random sampling (some users have only 1 rating)...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            random_state=42
        )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining PopRecommender model...")
    model = PopRecommender()
    model.fit(user_item_matrix)



    print("\nLoading local LLM model...")
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
        
    print("\nInitializing reranker (LLM Reranker)...")
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )

    
    rerankers = {
        "Original Pop": None,
        "Pop + LLMReranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Pop"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Pop":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Pop"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Pop":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of all relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in item recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")

    if isinstance(llm_reranker, LLMReranker):
        print("\n" + "="*30 + " LLM RERANKER LOG " + "="*30)
        print(f"Fallbacks wegen fehlender Metadaten: {llm_reranker.fallback_count}")
        print("Zielverteilungen laut LLM:")
        for goal, count in llm_reranker.goal_counter.items():
            print(f"- {goal}: {count}x")
    
    return all_results

#################################
# MAIN EXECUTION
#################################

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading LastFM dataset...
Loading LastFM dataset...
Loaded LastFM with 92834 interactions
Splitting data for evaluation...
Using random sampling (some users have only 1 rating)...
Creating user-item matrix...


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



Training PopRecommender model...
Popularity-based recommender ready! Top 5 most popular items: [152  52 279 105  13]

Loading local LLM model...

Initializing reranker (LLM Reranker)...

Evaluating 1875 users...

Evaluating Original Pop...

Evaluating Pop + LLMReranker...

============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original Pop        Pop + LLMReranker   
--------------------------------------------------------------------------------
ndcg@10        0.0970               0.0953 (-1.8%)     
precision@10   0.0721               0.0692 (-4.0%)     
recall@10      0.0779               0.0747 (-4.1%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original Pop        Pop + LLMReranker   
--------------------------------------------------------------------------------
item_coverage  0.0019               0.0029 (+51.7%)     
gini_index     0.5467               0.6793 (

***LastFM with Random, Reranking with LLAMA (2000 USER)***

In [12]:
# RANDOM Recommender with LLM-based Reranker for LastFM Dataset

import numpy as np
import pandas as pd
import random
from collections import defaultdict, Counter
import math
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from llama_cpp import Llama
import torch


# POP RECOMMENDER IMPLEMENTATION
class RandomRecommender:
    def __init__(self, random_state=42):
        self.random_state = random_state
        random.seed(random_state)
        np.random.seed(random_state)

    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
    
        self.user_items = defaultdict(set)
        for user, item in zip(*user_item_matrix.nonzero()):
            self.user_items[user].add(item)
    
        # Für Kompatibilität mit Reranker: zufällige Item-Faktoren
        np.random.seed(self.random_state)
        self.item_factors = np.random.normal(0, 0.1, (self.n_items, 32))
    
        # Gleichverteilte Popularität (alle Items sind gleich beliebt)
        self.item_popularity = np.ones(self.n_items)
    
        print(f"Random recommender ready! Total items: {self.n_items}")
        return self



    def recommend(self, user_id, n=10, exclude_seen=True):
        local_random = random.Random(self.random_state + user_id)
        all_items = list(range(self.n_items))
        
        if exclude_seen and user_id in self.user_items:
            candidate_items = [item for item in all_items if item not in self.user_items[user_id]]
        else:
            candidate_items = all_items
        
        if len(candidate_items) <= n:
            return np.array(candidate_items)
        
        recommended_items = local_random.sample(candidate_items, n)
        return np.array(recommended_items)
        



#################################
# LLM-BASED RERANKER FOR ITEMKNN
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Calculate item popularity for novelty/fairness
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

        self.fallback_count = 0
        self.goal_counter = Counter()

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]

        lines = [f"{i+1}. {item['title']}\nDescription: {item['description']}" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)

        prompt = (
            f"[INST] A recommender system suggests the following music artists to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Based on the artist descriptions, choose the most appropriate re-ranking goal for this user:\n"
            f"- accuracy (focus on most relevant artists)\n"
            f"- diverse_first (focus on a wide variety of music genres)\n"
            f"- fair_first (focus on less popular or underrepresented artists)\n"
            f"- balance (a mix of all of the above)\n\n"
            f"Return only the goal word. [/INST]"
        )

        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()

        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                self.goal_counter[goal] += 1
                return goal

        self.user_goal_map[user_id] = "balance"
        self.goal_counter["balance"] += 1
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)

        # Check if enough metadata is available
        valid_items = [
            item for item in candidates[:10]
            if self.reverse_item_mapping[item] in self.item_metadata
        ]
        if len(valid_items) < 5:
            self.fallback_count += 1
            return candidates[:n].tolist()

        # Get metadata
        top_items = [
            self.item_metadata[self.reverse_item_mapping[item]]
            for item in valid_items
        ]
        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        # Compute predicted scores depending on model type
        if hasattr(self.model, 'model') and hasattr(self.model.model, 'eval'):
            # NeuMF: use neural network prediction
            self.model.model.eval()
            with torch.no_grad():
                user_tensor = torch.LongTensor([user_id] * self.model.n_items).to(self.model.device)
                item_tensor = torch.LongTensor(list(range(self.model.n_items))).to(self.model.device)
                predicted_scores = self.model.model(user_tensor, item_tensor).cpu().numpy()
        else:
            # PopRecommender: use normalized popularity as proxy for accuracy score
            predicted_scores = self.norm_popularity.copy()


        selected = []
        while len(selected) < n and len(candidates) > 0:
            best_score = -np.inf
            best_item = None

            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = predicted_scores[item]

                if selected:
                    similarities = [
                        np.dot(self.model.item_factors[item], self.model.item_factors[sel_item]) /
                        (np.linalg.norm(self.model.item_factors[item]) * np.linalg.norm(self.model.item_factors[sel_item]) + 1e-10)
                        for sel_item in selected
                    ]
                    diversity_score = 1 - np.mean(similarities)
                else:
                    diversity_score = 1

                novelty_score = 1 - self.norm_popularity[item]

                combined_score = (w1 * score_accuracy +
                                  w2 * diversity_score +
                                  w3 * novelty_score)

                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item

            if best_item is None:
                break

            selected.append(best_item)
            candidates = candidates[candidates != best_item]

        return selected


#################################
# EVALUATION METRICS
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += rel / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += rel / np.log2(i + 2)
    if idcg == 0:
        return 0
    ndcg = dcg / idcg
    return ndcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    precision = num_relevant_recommended / len(recommended_items) if recommended_items else 0
    return precision

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    recall = num_relevant_recommended / len(relevant_items) if relevant_items else 0
    return recall

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = 0
        for i, count in enumerate(sorted_counts):
            cumulative_sum += (i + 1) * count
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS
#################################

def load_lastfm(path="lastfm/lastfm.inter"):
    print("Loading LastFM dataset...")
    for enc in ['utf-8','latin-1','ISO-8859-1','cp1252']:
        try:
            df = pd.read_csv(path, sep='\t',
                             names=['user_id','artist_id','weight','tag_value'],
                             encoding=enc)
            df = df.rename(columns={'artist_id': 'item_id', 'weight': 'rating'})
            df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
            df = df.dropna(subset=['rating'])
            df['rating'] = df['rating'].astype(float)
            df['timestamp'] = 1111111111
            print(f"Loaded LastFM with {len(df)} interactions")

            # ⬇️ generate_item_metadata erwartet tag_value in raw_df
            item_metadata = generate_item_metadata(df)
            return df[['user_id', 'item_id', 'rating', 'timestamp']], item_metadata
        except Exception:
            continue


def generate_item_metadata(raw_df):
    """
    Generates metadata for LastFM items based on available tags.

    Parameters:
    - raw_df: DataFrame with columns ['user_id', 'item_id', 'rating', 'timestamp', 'tag_value']

    Returns:
    - metadata: Dictionary with structure {item_id: {'title': str, 'genres': list, 'description': str}}
    """
    metadata = {}

    # Gruppiere nach item_id und kombiniere alle zugehörigen tag_value
    tag_groups = raw_df.groupby('item_id')['tag_value'].apply(lambda tags: ",".join(filter(pd.notna, tags)))

    for item_id, tags in tag_groups.items():
        tag_list = [tag.strip() for tag in tags.split(',') if tag.strip()]
        if not tag_list:
            continue  # Ignoriere Items ohne Tags
        metadata[item_id] = {
            'title': str(item_id),
            'genres': tag_list,
            'description': f"This artist is known for: {', '.join(tag_list[:10])}."  # Max 10 Tags in Prompt
        }

    return metadata


#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading LastFM dataset...")
    ratings_df, item_metadata = load_lastfm()
    
    print("Splitting data for evaluation...")
    value_counts = ratings_df['user_id'].value_counts()
    if value_counts.min() >= 2:
        print("Using stratified sampling...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            stratify=ratings_df['user_id'], 
            random_state=42
        )
    else:
        print("Using random sampling (some users have only 1 rating)...")
        train_df, test_df = train_test_split(
            ratings_df, 
            test_size=0.2, 
            random_state=42
        )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining RandomRecommender model...")
    model = RandomRecommender()
    model.fit(user_item_matrix)



    print("\nLoading local LLM model...")
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
        
    print("\nInitializing reranker (LLM Reranker)...")
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )

    
    rerankers = {
        "Original Random": None,
        "Random + LLMReranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Random"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Random":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Random"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Random":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of all relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in item recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")

    if isinstance(llm_reranker, LLMReranker):
        print("\n" + "="*30 + " LLM RERANKER LOG " + "="*30)
        print(f"Fallbacks wegen fehlender Metadaten: {llm_reranker.fallback_count}")
        print("Zielverteilungen laut LLM:")
        for goal, count in llm_reranker.goal_counter.items():
            print(f"- {goal}: {count}x")
    
    return all_results

#################################
# MAIN EXECUTION
#################################

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading LastFM dataset...
Loading LastFM dataset...
Loaded LastFM with 92834 interactions
Splitting data for evaluation...
Using random sampling (some users have only 1 rating)...
Creating user-item matrix...


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



Training RandomRecommender model...
Random recommender ready! Total items: 15394

Loading local LLM model...

Initializing reranker (LLM Reranker)...

Evaluating 1875 users...

Evaluating Original Random...

Evaluating Random + LLMReranker...

============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original Random     Random + LLMReranker
--------------------------------------------------------------------------------
ndcg@10        0.0002               0.0007 (+201.0%)     
precision@10   0.0004               0.0008 (+87.5%)     
recall@10      0.0005               0.0009 (+79.8%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original Random     Random + LLMReranker
--------------------------------------------------------------------------------
item_coverage  0.7014               0.7025 (+0.2%)     
gini_index     0.2658               0.2650 (-0.3%)     
shannon_entropy